In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
from pyspark.sql import functions as F

players = spark.table("lh_silver_game.players_clean")
sessions = spark.table("lh_silver_game.sessions_clean")
purchases = spark.table("lh_silver_game.purchases_clean")
ad_events = spark.table("lh_silver_game.ad_events_clean")
level_events = spark.table("lh_silver_game.level_events_clean")

StatementMeta(, 3bfba3b7-ce76-4686-a07f-9b2c35a275fc, 3, Finished, Available, Finished, False)

In [2]:
session_metrics = (
    sessions
    .groupBy("player_id")
    .agg(
        F.count("*").alias("total_sessions"),
        F.sum("session_duration_minutes").alias("total_session_minutes"),
        F.avg("session_duration_minutes").alias("avg_session_duration"),
        F.max("days_since_install").alias("max_days_since_install")
    )
)

StatementMeta(, 3bfba3b7-ce76-4686-a07f-9b2c35a275fc, 4, Finished, Available, Finished, False)

In [3]:
print("Player count:", session_metrics.count())
display(session_metrics.limit(5))

StatementMeta(, 3bfba3b7-ce76-4686-a07f-9b2c35a275fc, 5, Finished, Available, Finished, False)

Player count: 49988


SynapseWidget(Synapse.DataFrame, b357296a-ed47-4f10-9cc0-e48ae048bb28)

In [4]:
purchase_metrics = (
    purchases
    .groupBy("player_id")
    .agg(
        F.count("*").alias("total_purchases"),
        F.sum("price_usd").alias("total_purchase_revenue"),
        F.avg("price_usd").alias("avg_purchase_value")
    )
)

StatementMeta(, 3bfba3b7-ce76-4686-a07f-9b2c35a275fc, 6, Finished, Available, Finished, False)

In [5]:
display(purchase_metrics.limit(5))

StatementMeta(, 3bfba3b7-ce76-4686-a07f-9b2c35a275fc, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4de68f33-e8e1-44ae-9844-19723a7f6c4d)

In [6]:
ad_metrics = (
    ad_events.groupBy("player_id").agg(
        F.count("*").alias("total_ad_events"),
        F.sum("revenue_usd").alias("total_ad_revenue"),
        F.avg("revenue_usd").alias("avg_ad_revenue")
    )
)

StatementMeta(, 3bfba3b7-ce76-4686-a07f-9b2c35a275fc, 8, Finished, Available, Finished, False)

In [7]:
display(ad_metrics.limit(5))

StatementMeta(, 3bfba3b7-ce76-4686-a07f-9b2c35a275fc, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 68b6b6f5-c7e5-4b35-b9e6-3728cf6f9141)

In [8]:
level_metrics = (
    level_events
    .groupBy("player_id")
    .agg(
        F.sum(
            F.when(F.col("event_name") == "level_start", 1).otherwise(0)
        ).alias("total_level_starts"),

        F.sum(
            F.when(F.col("event_name") == "level_complete", 1).otherwise(0)
        ).alias("total_level_completes"),

        F.sum(
            F.when(F.col("event_name") == "level_fail", 1).otherwise(0)
        ).alias("total_level_fails"),

        F.max("level_number").alias("max_level_reached"),

        F.sum(
            F.when(F.col("booster_used") == True, 1).otherwise(0)
        ).alias("booster_usage_count")
    )
)

StatementMeta(, 3bfba3b7-ce76-4686-a07f-9b2c35a275fc, 10, Finished, Available, Finished, False)

In [9]:
print("Level player count:", level_metrics.count())
display(level_metrics.limit(5))

StatementMeta(, 3bfba3b7-ce76-4686-a07f-9b2c35a275fc, 11, Finished, Available, Finished, False)

Level player count: 50000


SynapseWidget(Synapse.DataFrame, 2cfa6c90-1a4b-490c-84f3-632a7627207e)

In [10]:
player_metrics = (
    players
    .join(session_metrics, on="player_id", how="left")
    .join(purchase_metrics, on="player_id", how="left")
    .join(ad_metrics, on="player_id", how="left")
    .join(level_metrics, on="player_id", how="left")
)

StatementMeta(, 3bfba3b7-ce76-4686-a07f-9b2c35a275fc, 12, Finished, Available, Finished, False)

In [11]:
player_metrics = player_metrics.fillna({
    "total_sessions": 0,
    "total_session_minutes": 0.0,
    "avg_session_duration": 0.0,
    "max_days_since_install": 0,

    "total_purchases": 0,
    "total_purchase_revenue": 0.0,
    "avg_purchase_value": 0.0,

    "total_ad_events": 0,
    "total_ad_revenue": 0.0,
    "avg_ad_revenue": 0.0,

    "total_level_starts": 0,
    "total_level_completes": 0,
    "total_level_fails": 0,
    "max_level_reached": 0,
    "booster_usage_count": 0
})

StatementMeta(, 3bfba3b7-ce76-4686-a07f-9b2c35a275fc, 13, Finished, Available, Finished, False)

In [12]:
print("Row count:", player_metrics.count())
print(
    "Unique players:",
    player_metrics.select("player_id").distinct().count()
)

display(player_metrics.limit(5))

StatementMeta(, 3bfba3b7-ce76-4686-a07f-9b2c35a275fc, 14, Finished, Available, Finished, False)

Row count: 50000
Unique players: 50000


SynapseWidget(Synapse.DataFrame, 8e3552bb-9d5f-4d59-9e14-31ac9a61d775)

In [13]:
player_metrics = (
    player_metrics
    .withColumn(
        "is_payer",
        F.when(F.col("total_purchases") > 0, 1).otherwise(0)
    )
    .withColumn(
        "total_revenue",
        F.col("total_purchase_revenue") + F.col("total_ad_revenue")
    )
    .withColumn(
        "level_fail_rate",
        F.when(
            (F.col("total_level_completes") + F.col("total_level_fails")) > 0,
            F.col("total_level_fails") /
            (F.col("total_level_completes") + F.col("total_level_fails"))
        ).otherwise(0)
    )
)

StatementMeta(, 3bfba3b7-ce76-4686-a07f-9b2c35a275fc, 15, Finished, Available, Finished, False)

In [14]:
display(
    player_metrics.select(
        "player_id",
        "total_sessions",
        "total_purchases",
        "is_payer",
        "total_purchase_revenue",
        "total_ad_revenue",
        "total_revenue",
        "max_level_reached",
        "level_fail_rate"
    ).limit(10)
)

StatementMeta(, 3bfba3b7-ce76-4686-a07f-9b2c35a275fc, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8279cb48-3727-40e2-89fa-9f8c17f726ac)

In [15]:
player_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lh_gold_game.player_metrics")

StatementMeta(, 3bfba3b7-ce76-4686-a07f-9b2c35a275fc, 17, Finished, Available, Finished, False)

In [16]:
df_check = spark.table("lh_gold_game.player_metrics")

print("Saved row count:", df_check.count())
display(df_check.limit(5))

StatementMeta(, 3bfba3b7-ce76-4686-a07f-9b2c35a275fc, 18, Finished, Available, Finished, False)

Saved row count: 50000


SynapseWidget(Synapse.DataFrame, 2d7ead83-54e6-4e89-b7fb-e072848af361)